# 03 - Feature Engineering

**Goal:** Turn raw weather columns into the exact inputs a model needs:

1. **The target** `RainTomorrow` (what we want to predict)
2. **Seasonal features** (Month, Season, DayOfYear)
3. **"Yesterday" lag features** (rain/temperature/humidity from the previous day)

And critically: **prove that no feature leaks future information.**

### The golden rule of this phase

A model predicting *tomorrow's* rain may only use information that exists
**before tomorrow happens**. We will build the target from tomorrow's rain,
build every feature from today's or *yesterday's* data, and then run an
explicit **leakage audit** to prove it.


## Setup

We start from the cleaned dataset produced in Phase 2.


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

PROCESSED = Path("data/processed/karachi_weather_clean.csv")
if not PROCESSED.exists():
    PROCESSED = Path("..") / "data/processed/karachi_weather_clean.csv"

df = pd.read_csv(PROCESSED, parse_dates=["Date"])
print("Rows:", len(df), "| Columns:", len(df.columns))

Rows: 15198 | Columns: 12


## 1. The target: RainTomorrow

We define a rainy day as **rainfall > 0 mm** (documented threshold).

`RainTomorrow` asks: *"is TOMORROW a rainy day?"*

To create it we look at each row's **next day** rainfall. In pandas that is
`shift(-1)`: shift *one row up* (towards the future).

```
Today's rainfall   Tomorrow's rainfall   RainTomorrow
  0 mm                 5 mm                 1 (Rain)
  0 mm                 0 mm                 0 (No rain)
```


In [2]:
df["RainToday"] = (df["Rainfall"] > 0).astype(int)

# RainTomorrow = is TOMORROW a rainy day?
# The very last row has no real "tomorrow", so its target stays NaN
# (unknown) instead of a fake 0. We drop that row later.
tomorrow_rain = df["Rainfall"].shift(-1)
df["RainTomorrow"] = (tomorrow_rain > 0).astype(float)
df.loc[tomorrow_rain.isna(), "RainTomorrow"] = np.nan

print("RainTomorrow distribution (NaN = last row, no real tomorrow):")
print(df["RainTomorrow"].value_counts(dropna=False).sort_index())
print()
print("RainTomorrow = 1 (rain):", int((df["RainTomorrow"] == 1).sum()), f"({df['RainTomorrow'].mean()*100:.1f}% of known days)")

RainTomorrow distribution (NaN = last row, no real tomorrow):
RainTomorrow
0.0    13174
1.0     2023
NaN        1
Name: count, dtype: int64

RainTomorrow = 1 (rain): 2023 (13.3% of known days)


### Why the threshold `> 0 mm`?

A single raindrop would make a day "rainy" technically, but Open-Meteo rounds
rainfall to 0.1 mm, so `Rainfall > 0` means "at least a trace of rain fell".
This is a simple, reproducible definition. Whatever threshold we choose must be
**documented** and used consistently - which we are doing here.


## 2. Verify the target logic

We manually inspect a few rows to confirm `RainTomorrow` really equals
*tomorrow's* `RainToday`. This kind of spot-check catches sign mistakes.


In [3]:
check = df[["Date", "Rainfall", "RainToday", "RainTomorrow"]].head(6).copy()
check["next_day_rain"] = df["RainToday"].shift(-1).head(6)
print(check.to_string(index=False))
print()
print("All correct:", bool((check["RainTomorrow"] == check["next_day_rain"]).all()))

      Date  Rainfall  RainToday  RainTomorrow  next_day_rain
1985-01-01       0.0          0           0.0            0.0
1985-01-02       0.0          0           0.0            0.0
1985-01-03       0.0          0           0.0            0.0
1985-01-04       0.0          0           1.0            1.0
1985-01-05       1.3          1           0.0            0.0
1985-01-06       0.0          0           1.0            1.0

All correct: True


## 3. Seasonal features

Karachi's rain is strongly seasonal (monsoon in Jul-Aug). The model does not
know months by itself, so we hand it the calendar info:

- **Month** (1-12): captures the monthly rhythm
- **DayOfYear** (1-365): continuous version of the same idea
- **Season** (categorical): groups months into weather regimes

Season mapping used for Karachi:
| Season | Months |
|--------|--------|
| Winter | Dec, Jan, Feb |
| Hot Dry | Mar, Apr, May |
| Monsoon | Jun, Jul, Aug, Sep |
| Post-Monsoon | Oct, Nov |


In [4]:
def season_of(month: int) -> str:
    if month in (12, 1, 2):
        return "Winter"
    if month in (3, 4, 5):
        return "HotDry"
    if month in (6, 7, 8, 9):
        return "Monsoon"
    return "PostMonsoon"

df["Month"] = df["Date"].dt.month
df["DayOfYear"] = df["Date"].dt.dayofyear
df["Season"] = df["Month"].map(season_of)

print("Season counts:")
print(df["Season"].value_counts())

Season counts:
Season
Monsoon        5074
HotDry         3864
Winter         3759
PostMonsoon    2501
Name: count, dtype: int64


## 4. "Yesterday" lag features

Weather patterns persist: if it rained **today**, it is more likely to rain
**tomorrow** (monsoon systems move slowly). So yesterday's and today's
observations are useful.

Lags are made with `shift(1)`: shift one row *down* (towards the past).
These use only **past** information - safe to use as features.


In [5]:
df["PreviousRainfall"] = df["Rainfall"].shift(1)          # rain yesterday (mm)
df["PreviousDayTemperature"] = df["MeanTemperature"].shift(1)  # temp yesterday (C)
df["PreviousDayHumidity"] = df["Humidity"].shift(1)            # humidity yesterday (%)

print("Sample of lag features:")
print(df[["Date", "Rainfall", "PreviousRainfall", "MeanTemperature",
          "PreviousDayTemperature", "Humidity", "PreviousDayHumidity"]].head(4).to_string(index=False))

Sample of lag features:
      Date  Rainfall  PreviousRainfall  MeanTemperature  PreviousDayTemperature  Humidity  PreviousDayHumidity
1985-01-01       0.0               NaN             16.4                     NaN        63                  NaN
1985-01-02       0.0               0.0             16.7                    16.4        57                 63.0
1985-01-03       0.0               0.0             17.1                    16.7        44                 57.0
1985-01-04       0.0               0.0             17.7                    17.1        50                 44.0


### Why do lag features help?

- **Persistence:** rain and storms often continue across consecutive days.
- **Context:** yesterday's humidity/temperature describes the *trend* of the
  atmosphere, not just a single snapshot.

Note the first row has `NaN` lags (no "yesterday") - we handle that below.


## 5. Remove a redundant column

`Precipitation` and `Rainfall` are identical (Karachi gets no snow or hail,
so all precipitation is rain). Keeping both would give the model two copies of
the same information. We drop `Precipitation` and document why.


In [6]:
same = (df["Precipitation"] == df["Rainfall"]).all()
print("Precipitation == Rainfall for all rows:", same)
df = df.drop(columns=["Precipitation"])
print("Columns now:", list(df.columns))

Precipitation == Rainfall for all rows: True
Columns now: ['Date', 'MaxTemperature', 'MinTemperature', 'MeanTemperature', 'Rainfall', 'Pressure', 'Humidity', 'CloudCoverage', 'WindSpeed', 'WindDirection', 'WeatherCode', 'RainToday', 'RainTomorrow', 'Month', 'DayOfYear', 'Season', 'PreviousRainfall', 'PreviousDayTemperature', 'PreviousDayHumidity']


## 6. THE LEAKAGE AUDIT (most important cell in this notebook)

**Leakage** means a feature accidentally contains future information, letting
the model "cheat" (it looks great in tests, but fails in real life).

Three checks prove our features are clean:

1. **Target isolation:** `RainTomorrow` must be the ONLY column built from
   tomorrow's data. We confirm it equals `RainToday.shift(-1)`.
2. **No feature equals any future observation:** for every feature and every
   raw column, there must be at least one row where they differ from
   `raw_column.shift(-1)` (the future value). If any feature was identical to
   a future value, it would leak.
3. **Lags are past, not future:** `PreviousRainfall` must equal yesterday's
   rainfall, i.e. `Rainfall.shift(1)`, never `shift(-1)`.


In [7]:
# --- check 1: target isolation ---
assert (df["RainToday"].shift(-1) == df["RainTomorrow"]).iloc[:-1].all()
print("[OK] RainTomorrow equals tomorrow's RainToday (target is correct)")

# --- check 2: no feature is identical to any future observation ---
raw_columns = ["MaxTemperature", "MinTemperature", "MeanTemperature",
               "Pressure", "Humidity", "CloudCoverage", "WindSpeed",
               "WindDirection", "Rainfall", "WeatherCode"]
feature_columns = [c for c in df.columns
                   if c not in ("Date", "RainTomorrow", "RainToday", "Month",
                                "DayOfYear", "Season")]
leaks = []
for feature in feature_columns:
    for raw_col in raw_columns:
        future = df[raw_col].shift(-1)
        # compare all rows except the last (future has no value there)
        if (df[feature] == future).iloc[:-1].all():
            leaks.append((feature, raw_col))
if leaks:
    print("LEAKS FOUND:", leaks)
else:
    print("[OK] no feature is identical to any future (shift -1) observation")

# --- check 3: lags point to the past, not the future ---
assert (df["PreviousRainfall"] == df["Rainfall"].shift(1)).iloc[1:].all(), "PreviousRainfall leak!"
assert (df["PreviousDayTemperature"] == df["MeanTemperature"].shift(1)).iloc[1:].all()
assert (df["PreviousDayHumidity"] == df["Humidity"].shift(1)).iloc[1:].all()
print("[OK] all lag features equal their PAST (shift +1) source columns")

print()
print("ALL LEAKAGE CHECKS PASSED")

[OK] RainTomorrow equals tomorrow's RainToday (target is correct)


[OK] no feature is identical to any future (shift -1) observation


[OK] all lag features equal their PAST (shift +1) source columns

ALL LEAKAGE CHECKS PASSED


### What this audit proves

- `RainTomorrow` is derived from tomorrow, everything else is derived from
  today or yesterday.
- If we had accidentally added e.g. `TomorrowTemperature`, check 2 would have
  caught it immediately.
- This is the safety net that makes our evaluation trustworthy later.


## 7. Handle the edge rows

Two rows cannot be used:

- The **first** row: its lag features are `NaN` (no "yesterday").
- The **last** row: it has no "tomorrow", so `RainTomorrow` is `NaN`.

We drop these two rows. Removing 2 out of 15,198 rows is negligible and is
the honest choice - we never fabricate values for missing neighbors.


In [8]:
print("Rows before:", len(df))
print("Rows with missing target:", int(df["RainTomorrow"].isna().sum()))
print("Rows with missing lags:  ", int(df[["PreviousRainfall", "PreviousDayTemperature", "PreviousDayHumidity"]].isna().any(axis=1).sum()))

df = df.dropna(subset=["RainTomorrow", "PreviousRainfall"]).reset_index(drop=True)
print("Rows after:", len(df))
print("Any remaining missing values:", int(df.isna().sum().sum()))

Rows before: 15198
Rows with missing target: 1
Rows with missing lags:   1
Rows after: 15196
Any remaining missing values: 0


## 8. Quick look at the engineered dataset

Show the final shape and a peek at the columns we will feed the model.


In [9]:
print("Final shape:", df.shape)
print("Target balance -> RainTomorrow = 1:", f"{(df['RainTomorrow'].mean()*100):.1f}%")
print()
print(df.head(3).to_string())

Final shape: (15196, 19)
Target balance -> RainTomorrow = 1: 13.3%

        Date  MaxTemperature  MinTemperature  MeanTemperature  Rainfall  Pressure  Humidity  CloudCoverage  WindSpeed  WindDirection  WeatherCode  RainToday  RainTomorrow  Month  DayOfYear  Season  PreviousRainfall  PreviousDayTemperature  PreviousDayHumidity
0 1985-01-02            21.7            12.2             16.7       0.0    1019.0        57              1       19.1             69            0          0           0.0      1          2  Winter               0.0                    16.4                 63.0
1 1985-01-03            22.2            12.5             17.1       0.0    1019.2        44              1       17.9             57            0          0           0.0      1          3  Winter               0.0                    16.7                 57.0
2 1985-01-04            22.4            12.8             17.7       0.0    1017.0        50             10       12.3            352            1       

## 9. Save the engineered dataset

Saved to `data/processed/karachi_weather_features.csv` for the training phase.


In [10]:
OUT = Path("data/processed/karachi_weather_features.csv")
if not OUT.parent.exists():
    OUT.parent.mkdir(parents=True)
df.to_csv(OUT, index=False)
print(f"Saved {len(df)} rows x {df.shape[1]} cols -> {OUT}")

Saved 15196 rows x 19 cols -> data\processed\karachi_weather_features.csv


## Summary of Phase 3

| Feature | Type | Based on | Leakage? |
|---------|------|----------|----------|
| `RainTomorrow` | target | tomorrow's rainfall | is the target, not a feature |
| `Month`, `Season`, `DayOfYear` | seasonal | calendar date | none (calendar known in advance) |
| `PreviousRainfall` | lag | yesterday's rain | none (past) |
| `PreviousDayTemperature` | lag | yesterday's temp | none (past) |
| `PreviousDayHumidity` | lag | yesterday's humidity | none (past) |
| all today's columns | direct | today's observation | none (known by end of today) |

The leakage audit passed all three checks. We are now ready for modeling.

**Next phase:** Phase 4 - Chronological train/test split, then a Logistic
Regression baseline.
